In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Trening modelu

In [ ]:
!pip install -U ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.3.230 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 37.9/112.6 GB disk)


In [ ]:
from ultralytics import YOLO

## Wczytanie danych

In [ ]:
zip_path = "/content/drive/MyDrive/dane_detekcja_pojazdow/EAGLE_dataset_1024_res_cleaned_obb.zip" #mniejszy zbiór

!cp "{zip_path}" /content/

In [ ]:
!unzip -q /content/EAGLE_dataset_1024_res_cleaned_obb.zip -d /content/data

In [ ]:
from pathlib import Path
import os
import yaml

data_path = Path("/content/data/content/data_1024_res")
yaml_file_path = data_path / "data.yaml"

# Utworzenie pliku YAML

In [ ]:
data_content = {
    'path': '../EagleDatasetYOLO',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,
    'names': ['vehicle']
}

with open(yaml_file_path, 'w') as f:
    yaml.dump(data_content, f, default_flow_style=False)

print(f"YAML file created at: {yaml_file_path}")
print("Content:")
print(yaml.dump(data_content, indent=2))

YAML file created at: /content/data/content/data_1024_res/data.yaml
Content:
names:
- vehicle
nc: 1
path: ../EagleDatasetYOLO
test: images/test
train: images/train
val: images/val



## Poprawa ścieżki path w YAML

In [ ]:

with open(yaml_file_path, 'r') as f:
    yaml_content = yaml.safe_load(f)

print(yaml.dump(yaml_content, indent=2))

names:
- vehicle
nc: 1
path: ../EagleDatasetYOLO
test: images/test
train: images/train
val: images/val



In [ ]:

# Assuming data_path and yaml_file_path are already defined
# (based on previous notebook state)

# Ensure data_path is a string for the YAML content
updated_data_path_str = str(data_path)

# Update the 'path' attribute in the loaded YAML content
yaml_content['path'] = updated_data_path_str

# Save the updated YAML content back to the file
with open(yaml_file_path, 'w') as f:
    yaml.dump(yaml_content, f)

print(f"'path' attribute in {yaml_file_path} updated to: {yaml_content['path']}")

# Display the updated YAML content to verify
print("\nUpdated YAML content:")
print(yaml.dump(yaml_content, indent=2))

'path' attribute in /content/data/content/data_1024_res/data.yaml updated to: /content/data/content/data_1024_res

Updated YAML content:
names:
- vehicle
nc: 1
path: /content/data/content/data_1024_res
test: images/test
train: images/train
val: images/val



## Trening

In [ ]:
model = YOLO("yolo11n-obb.pt")
results = model.train(
    data=yaml_file_path,
    epochs=50,
    imgsz=1024,
    batch=8,
    patience=10,
    single_cls=True,

    # do zapisu jbkc
    project='/content/drive/MyDrive/YOLO_Project',
    name='test_scale_1_0', # Nadaj nazwę, żeby łatwo znaleźć folder

    save_period=1,  # Zapisuj checkpoint co każdą epokę (dla bezpieczeństwa)
    exist_ok=True   # Nie wywalaj błędu, jeśli folder już istnieje
    )

Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/content/data_1024_res/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=test_scale_1_0, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspec

# Wznawianie treningu

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      18/50      9.94G     0.8529     0.5678      1.158        108       1024: 100% ━━━━━━━━━━━━ 550/550 1.9it/s 4:53
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.3it/s 40.9s
                   all       1397      47537      0.955      0.907      0.968      0.778

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      19/50      6.21G     0.8498     0.5625      1.172        193       1024: 100% ━━━━━━━━━━━━ 550/550 2.0it/s 4:29
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 5.0it/s 35.1s
                   all       1397      47537      0.953      0.906      0.967      0.778

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      20/50      8.57G     0.8468      0.561      1.163        304       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:22
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.9it/s 35.7s
                   all       1397      47537      0.952      0.909      0.967      0.782

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      21/50      7.79G     0.8397     0.5549      1.156        258       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 5.1it/s 34.3s
                   all       1397      47537      0.953      0.913      0.969      0.778

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      22/50      7.22G     0.8343     0.5492      1.159        111       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.7it/s 37.1s
                   all       1397      47537      0.952      0.911      0.969      0.776

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      23/50      8.96G     0.8335     0.5492      1.158         57       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:21
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 5.0it/s 34.7s
                   all       1397      47537      0.951      0.913      0.969      0.777

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      24/50      7.41G     0.8316     0.5466      1.151        113       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:21
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.9it/s 35.5s
                   all       1397      47537       0.95      0.915      0.969      0.782

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      25/50      8.93G     0.8212     0.5365      1.154        112       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:19
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.8it/s 36.6s
                   all       1397      47537      0.952      0.912      0.969      0.783

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      26/50      8.46G     0.8169     0.5352      1.148        157       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:21
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 5.1it/s 34.1s
                   all       1397      47537      0.956      0.911       0.97      0.777

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      27/50      5.83G     0.8187     0.5352      1.153        208       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:21
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.7it/s 37.1s
                   all       1397      47537      0.958      0.909      0.969      0.787

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      28/50      8.82G     0.8099     0.5304      1.147        240       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:25
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 5.1it/s 34.2s
                   all       1397      47537      0.956      0.912      0.969       0.78

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      29/50      6.44G     0.8174     0.5297      1.146        431       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.7it/s 36.8s
                   all       1397      47537      0.954      0.914       0.97      0.781

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      30/50      8.63G     0.8132     0.5271       1.14        297       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:20
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.9it/s 35.9s
                   all       1397      47537      0.955      0.914       0.97       0.78

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      31/50      6.06G     0.8021     0.5225      1.135        119       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:22
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 5.0it/s 35.2s
                   all       1397      47537      0.952      0.915       0.97      0.781

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      32/50      7.62G     0.8014     0.5206      1.141        170       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:21
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.9it/s 36.0s
                   all       1397      47537      0.956      0.916       0.97      0.789

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      33/50      7.78G     0.8031     0.5201      1.137        220       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.9it/s 35.6s
                   all       1397      47537      0.954      0.915       0.97      0.792

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      34/50      7.57G     0.8003     0.5191      1.137        423       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:28
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.9it/s 36.0s
                   all       1397      47537      0.955      0.918      0.971      0.786

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      35/50      9.19G     0.7938     0.5134      1.143        588       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.7it/s 36.9s
                   all       1397      47537      0.954      0.918      0.971      0.787

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      36/50       6.1G     0.8008     0.5198      1.138        177       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 5.0it/s 35.2s
                   all       1397      47537      0.956      0.912       0.97      0.787

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      37/50      6.12G     0.7975     0.5163      1.141         67       1024: 100% ━━━━━━━━━━━━ 550/550 2.1it/s 4:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 175/175 4.8it/s 36.7s
                   all       1397      47537      0.954      0.916      0.971       0.79

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      38/50      6.12G     0.7831     0.5065      1.133        327       1024: 39% ━━━━╸─────── 216/550 2.2it/s 1:43<2:30

In [ ]:
# Wskazujesz plik 'last.pt' z Twojego Google Drive
model = YOLO('/content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/last.pt')

# Wznawiasz trening
model.train(resume=True)

Ultralytics 8.3.230 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/content/data_1024_res/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=test_scale_1_0, nbs=64, nms=False, opset=None, optimize=False, opt

ultralytics.utils.metrics.OBBMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e64af1d1e20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 